In [4]:
import os
import json
import glob

# ==============================================================================
# 1. 설정
# ==============================================================================
base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data"

# 이름을 알고 싶은 ID들
target_ids = ["K-003351", "K-020014", "K-032310", "K-033880", "K-004543"]

print(f"🕵️‍♂️ 데이터셋 내부에서 약 이름(Metadata)을 수색합니다...\n")

# ==============================================================================
# 2. 단서 찾기 (JSON 뒤지기)
# ==============================================================================
# 모든 JSON 파일 가져오기 (시간이 좀 걸릴 수 있음)
all_jsons = glob.glob(os.path.join(base_path, "**", "*.json"), recursive=True)

# 결과를 저장할 사전
id_to_name = {}

for tid in target_ids:
    found_name = None
    
    # 해당 ID가 포함된 JSON 파일을 찾음
    for json_path in all_jsons:
        if tid in os.path.basename(json_path):
            try:
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    
                    # [단서 1] categories 안에 'name'이나 'class_name'이 있는지 확인
                    if "categories" in data:
                        for cat in data["categories"]:
                            # 보통 여기에 '타이레놀' 같은 이름이 들어있음
                            if "name" in cat and cat["name"] != "pill": 
                                found_name = cat["name"]
                                break
                            if "class_name" in cat:
                                found_name = cat["class_name"]
                                break
                                
                    # [단서 2] attributes 안에 있는지 확인
                    if not found_name and "attributes" in data:
                        if "class" in data["attributes"]:
                            found_name = data["attributes"]["class"]
                            
                    # [단서 3] 메타데이터(Meta) 정보 확인
                    if not found_name and "Meta" in data:
                        if "dl_name" in data["Meta"]: # 약 이름 (Drug Label Name)
                            found_name = data["Meta"]["dl_name"]
                            
                if found_name:
                    break # 이름을 찾았으면 중단
            except:
                continue
    
    id_to_name[tid] = found_name if found_name else "이름 정보 없음 (폴더명 확인 필요)"

# ==============================================================================
# 3. 결과 출력
# ==============================================================================
print("=" * 50)
print(f"📄 [알약 신원 조회 결과]")
print("=" * 50)

for tid in target_ids:
    name = id_to_name.get(tid, "찾지 못함")
    print(f"💊 ID: {tid}")
    print(f"   👉 이름: {name}")
    print("-" * 30)

print("\n💡 만약 '이름 정보 없음'이라고 뜬다면, 폴더 이름 자체가 약 이름일 수 있습니다.")

🕵️‍♂️ 데이터셋 내부에서 약 이름(Metadata)을 수색합니다...

📄 [알약 신원 조회 결과]
💊 ID: K-003351
   👉 이름: 일양하이트린정 2mg
------------------------------
💊 ID: K-020014
   👉 이름: 이름 정보 없음 (폴더명 확인 필요)
------------------------------
💊 ID: K-032310
   👉 이름: 이름 정보 없음 (폴더명 확인 필요)
------------------------------
💊 ID: K-033880
   👉 이름: 이름 정보 없음 (폴더명 확인 필요)
------------------------------
💊 ID: K-004543
   👉 이름: 이름 정보 없음 (폴더명 확인 필요)
------------------------------

💡 만약 '이름 정보 없음'이라고 뜬다면, 폴더 이름 자체가 약 이름일 수 있습니다.


In [3]:
import cv2
import json
import os

# ==========================================
# [설정] 본인 컴퓨터 경로에 맞게 수정하세요!
# ==========================================
base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data" 

# 누락된 파일 명단
missing_files = [
    {
        "img_rel": r"train_images\K-003351-013900-021325_0_2_0_2_70_000_200.png",
        "json_rel": r"train_annotations\K-003351-013900-021325_json\K-003351\K-003351-013900-021325_0_2_0_2_70_000_200.json",
        "target_pill": "K-003351"
    },
    {
        "img_rel": r"train_images\K-003351-013900-036637_0_2_0_2_70_000_200.png",
        "json_rel": r"train_annotations\K-003351-013900-036637_json\K-003351\K-003351-013900-036637_0_2_0_2_70_000_200.json",
        "target_pill": "K-003351"
    },
    {
        "img_rel": r"train_images\K-003351-020014-022074_0_2_0_2_90_000_200.png",
        "json_rel": r"train_annotations\K-003351-020014-022074_json\K-020014\K-003351-020014-022074_0_2_0_2_90_000_200.json",
        "target_pill": "K-020014"
    },
    {
        "img_rel": r"train_images\K-003351-021325-032310_0_2_0_2_90_000_200.png",
        "json_rel": r"train_annotations\K-003351-021325-032310_json\K-032310\K-003351-021325-032310_0_2_0_2_90_000_200.json",
        "target_pill": "K-032310"
    },
    {
        "img_rel": r"train_images\K-003351-032310-038162_0_2_0_2_70_000_200.png",
        "json_rel": r"train_annotations\K-003351-032310-038162_json\K-003351\K-003351-032310-038162_0_2_0_2_70_000_200.json",
        "target_pill": "K-003351"
    },
    {
        "img_rel": r"train_images\K-003351-033880-038162_0_2_0_2_75_000_200.png",
        "json_rel": r"train_annotations\K-003351-033880-038162_json\K-033880\K-003351-033880-038162_0_2_0_2_75_000_200.json",
        "target_pill": "K-033880"
    },
    {
        "img_rel": r"train_images\K-003351-035206-041768_0_2_0_2_70_000_200.png",
        "json_rel": r"train_annotations\K-003351-035206-041768_json\K-003351\K-003351-035206-041768_0_2_0_2_70_000_200.json",
        "target_pill": "K-003351"
    },
    {
        "img_rel": r"train_images\K-003544-004543-012247-016548_0_2_0_2_90_000_200.png",
        "json_rel": r"train_annotations\K-003544-004543-012247-016548_json\K-004543\K-003544-004543-012247-016548_0_2_0_2_90_000_200.json",
        "target_pill": "K-004543"
    }
]

# 전역 변수
drawing = False
ix, iy = -1, -1
bbox = []
img = None # 이미지 변수 전역화

def draw_rectangle(event, x, y, flags, param):
    global ix, iy, drawing, bbox, img
    
    if event == cv2.EVENT_LBUTTONDOWN: # 마우스 누름
        drawing = True
        ix, iy = x, y
        
    elif event == cv2.EVENT_LBUTTONUP: # 마우스 뗌
        drawing = False
        w = abs(x - ix)
        h = abs(y - iy)
        # BBox 저장 형식: [x시작, y시작, 가로길이, 세로길이]
        bbox = [min(ix, x), min(iy, y), w, h]
        print(f"✅ 좌표 잡힘: {bbox}")
        
        # 화면에 그리기
        cv2.rectangle(img, (min(ix, x), min(iy, y)), (min(ix, x)+w, min(iy, y)+h), (0, 255, 0), 2)
        cv2.imshow('Image', img)

print("=== 작업을 시작합니다 (총 8개) ===")
print("🖱️ 마우스 드래그: 박스 그리기")
print("⌨️ s 키: 저장하고 다음으로")
print("⌨️ r 키: (실수했을 때) 다시 그리기")
print("⌨️ q 키: 강제 종료")

# ... (앞부분 동일) ...

for i, item in enumerate(missing_files):
    img_path = os.path.join(base_path, item["img_rel"])
    save_path = os.path.join(base_path, item["json_rel"])
    target_pill = item["target_pill"]
    
    # 1. 경로 한글 깨짐 방지 및 존재 확인
    original_img = cv2.imread(img_path)
    if original_img is None:
        print(f"❌ 이미지를 불러올 수 없습니다: {img_path}")
        continue

    img = original_img.copy()
    bbox = []   # bbox 초기화

    # 2. 창 설정 (이름을 고정하고 속성 부여)
    win_name = f'Work [{i+1}/8] - target: {target_pill}'
    cv2.namedWindow(win_name, cv2.WINDOW_NORMAL) # 창 크기 조절 가능하게
    cv2.setMouseCallback(win_name, draw_rectangle)
    
    print(f"\n[{i+1}/8] {target_pill} 작업 중... (s: 저장, r: 리셋, q: 종료)")

    while True:
        cv2.imshow(win_name, img) # 같은 창 이름 사용
        k = cv2.waitKey(30) & 0xFF # 30ms 정도 여유를 줌
        
        if k == ord('s'):
            if not bbox:
                print("⚠️ 박스를 먼저 그려주세요!")
                continue
            
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            json_data = {
                "file_name": os.path.basename(img_path),
                "bbox": bbox, 
                "drug_id": target_pill
            }
            with open(save_path, 'w', encoding='utf-8') as f:
                json.dump(json_data, f, indent=4, ensure_ascii=False)
                
            print(f"💾 {target_pill} 저장 완료!")
            break

        elif k == ord('r'):
            print("🔄 리셋")
            img = original_img.copy()
            bbox = []
            
        elif k == ord('q'):
            print("작업 중단")
            cv2.destroyAllWindows()
            exit()
            
    cv2.destroyWindow(win_name) # 다음 파일을 위해 현재 창 닫기

cv2.destroyAllWindows()
print("\n🎉 모든 작업이 끝났습니다!")

=== 작업을 시작합니다 (총 8개) ===
🖱️ 마우스 드래그: 박스 그리기
⌨️ s 키: 저장하고 다음으로
⌨️ r 키: (실수했을 때) 다시 그리기
⌨️ q 키: 강제 종료

[1/8] K-003351 작업 중... (s: 저장, r: 리셋, q: 종료)
✅ 좌표 잡힘: [381, 832, 198, 190]
💾 K-003351 저장 완료!

[2/8] K-003351 작업 중... (s: 저장, r: 리셋, q: 종료)
✅ 좌표 잡힘: [428, 859, 196, 198]
💾 K-003351 저장 완료!

[3/8] K-020014 작업 중... (s: 저장, r: 리셋, q: 종료)
✅ 좌표 잡힘: [56, 720, 339, 318]
💾 K-020014 저장 완료!

[4/8] K-032310 작업 중... (s: 저장, r: 리셋, q: 종료)
✅ 좌표 잡힘: [576, 823, 368, 249]
💾 K-032310 저장 완료!

[5/8] K-003351 작업 중... (s: 저장, r: 리셋, q: 종료)
✅ 좌표 잡힘: [47, 245, 273, 225]
💾 K-003351 저장 완료!

[6/8] K-033880 작업 중... (s: 저장, r: 리셋, q: 종료)
✅ 좌표 잡힘: [60, 584, 303, 447]
💾 K-033880 저장 완료!

[7/8] K-003351 작업 중... (s: 저장, r: 리셋, q: 종료)
✅ 좌표 잡힘: [442, 872, 222, 197]
💾 K-003351 저장 완료!

[8/8] K-004543 작업 중... (s: 저장, r: 리셋, q: 종료)
✅ 좌표 잡힘: [624, 187, 227, 221]
💾 K-004543 저장 완료!

🎉 모든 작업이 끝났습니다!


잘 생성되었는지 확인 코드

In [2]:
import cv2
import json
import os
import numpy as np

# ==========================================
# [설정] 경로가 맞는지 다시 한 번 확인하세요!
# ==========================================
base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data" 

# 확인해야 할 파일 목록 (작업했던 리스트 그대로)
missing_files = [
    { "img_rel": r"train_images\K-003351-013900-021325_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-013900-021325_json\K-003351\K-003351-013900-021325_0_2_0_2_70_000_200.json", "target_pill": "K-003351" },
    { "img_rel": r"train_images\K-003351-013900-036637_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-013900-036637_json\K-003351\K-003351-013900-036637_0_2_0_2_70_000_200.json", "target_pill": "K-003351" },
    { "img_rel": r"train_images\K-003351-020014-022074_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003351-020014-022074_json\K-020014\K-003351-020014-022074_0_2_0_2_90_000_200.json", "target_pill": "K-020014" },
    { "img_rel": r"train_images\K-003351-021325-032310_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003351-021325-032310_json\K-032310\K-003351-021325-032310_0_2_0_2_90_000_200.json", "target_pill": "K-032310" },
    { "img_rel": r"train_images\K-003351-032310-038162_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-032310-038162_json\K-003351\K-003351-032310-038162_0_2_0_2_70_000_200.json", "target_pill": "K-003351" },
    { "img_rel": r"train_images\K-003351-033880-038162_0_2_0_2_75_000_200.png", "json_rel": r"train_annotations\K-003351-033880-038162_json\K-033880\K-003351-033880-038162_0_2_0_2_75_000_200.json", "target_pill": "K-033880" },
    { "img_rel": r"train_images\K-003351-035206-041768_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-035206-041768_json\K-003351\K-003351-035206-041768_0_2_0_2_70_000_200.json", "target_pill": "K-003351" },
    { "img_rel": r"train_images\K-003544-004543-012247-016548_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003544-004543-012247-016548_json\K-004543\K-003544-004543-012247-016548_0_2_0_2_90_000_200.json", "target_pill": "K-004543" }
]

print("=== 검수를 시작합니다 (아무 키나 누르면 다음 장으로 넘어갑니다) ===")

for i, item in enumerate(missing_files):
    img_path = os.path.join(base_path, item["img_rel"])
    json_path = os.path.join(base_path, item["json_rel"])
    
    # 1. JSON 파일이 있는지 확인
    if not os.path.exists(json_path):
        print(f"❌ [{i+1}] JSON 파일이 없습니다! 저장 안 됨: {item['target_pill']}")
        continue
        
    # 2. JSON 내용 읽기
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        bbox = data.get("bbox") # 저장된 bbox 가져오기
    
    # 3. 이미지 불러오기
    try:
        img_array = np.fromfile(img_path, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    except:
        print(f"❌ 이미지 로드 실패")
        continue

    # 4. 이미지 위에 박스 그리기 (파란색)
    if bbox:
        x, y, w, h = bbox
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2) # 파란색 두께 2
        
        # 텍스트 표시
        text = f"{item['target_pill']} (Check)"
        cv2.putText(img, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
    else:
        print(f"⚠️ [{i+1}] bbox 데이터가 비어있습니다!")

    # 5. 보여주기
    win_name = f"Check Result {i+1}/8"
    cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)
    cv2.imshow(win_name, img)
    
    print(f"✅ [{i+1}] {item['target_pill']} 확인 중... (스페이스바/엔터 누르면 다음)")
    cv2.waitKey(0) # 키 입력 대기 (무한 대기)
    cv2.destroyWindow(win_name)

cv2.destroyAllWindows()
print("🎉 검수 완료!")

=== 검수를 시작합니다 (아무 키나 누르면 다음 장으로 넘어갑니다) ===
✅ [1] K-003351 확인 중... (스페이스바/엔터 누르면 다음)
✅ [2] K-003351 확인 중... (스페이스바/엔터 누르면 다음)
✅ [3] K-020014 확인 중... (스페이스바/엔터 누르면 다음)
✅ [4] K-032310 확인 중... (스페이스바/엔터 누르면 다음)
✅ [5] K-003351 확인 중... (스페이스바/엔터 누르면 다음)
✅ [6] K-033880 확인 중... (스페이스바/엔터 누르면 다음)
✅ [7] K-003351 확인 중... (스페이스바/엔터 누르면 다음)
✅ [8] K-004543 확인 중... (스페이스바/엔터 누르면 다음)
🎉 검수 완료!


문제 파일 수정

In [ ]:
# import cv2
# import json
# import os
# import numpy as np

# # ==========================================
# # [설정] 경로가 맞는지 확인해주세요!
# # ==========================================
# base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data" 

# # 문제가 된 파일 (K-020014) 정보 직접 지정
# target_info = {
#     "img_rel": r"train_images\K-003351-020014-022074_0_2_0_2_90_000_200.png",
#     "json_rel": r"train_annotations\K-003351-020014-022074_json\K-020014\K-003351-020014-022074_0_2_0_2_90_000_200.json",
#     "target_pill": "K-020014"
# }

# # 전역 변수
# drawing = False
# ix, iy = -1, -1
# bbox = []
# img = None 
# original_img = None

# def draw_rectangle(event, x, y, flags, param):
#     global ix, iy, drawing, bbox, img, original_img
    
#     if event == cv2.EVENT_LBUTTONDOWN:
#         drawing = True
#         ix, iy = x, y
        
#     elif event == cv2.EVENT_LBUTTONUP:
#         drawing = False
#         w = abs(x - ix)
#         h = abs(y - iy)
        
#         if w < 10 or h < 10: return 

#         bbox = [min(ix, x), min(iy, y), w, h]
#         print(f"✅ 좌표 잡힘: {bbox}")
        
#         temp_img = original_img.copy()
#         cv2.rectangle(temp_img, (bbox[0], bbox[1]), (bbox[0]+w, bbox[1]+h), (0, 255, 0), 2)
#         img = temp_img 

# print("=== 🚑 긴급 복구 모드: K-020014 ===")

# img_path = os.path.join(base_path, target_info["img_rel"])
# save_path = os.path.join(base_path, target_info["json_rel"])
# target_pill = target_info["target_pill"]

# # 이미지 로드
# try:
#     img_array = np.fromfile(img_path, np.uint8)
#     original_img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
# except:
#     print("❌ 이미지를 찾을 수 없습니다. 경로를 확인하세요.")
#     exit()

# if original_img is None:
#     print("❌ 이미지가 로드되지 않았습니다.")
#     exit()

# img = original_img.copy()
# bbox = [] 

# # 창 띄우기 설정
# win_name = f'FORCE FIX: {target_pill}'
# cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)
# cv2.setMouseCallback(win_name, draw_rectangle)

# print(f"📸 창이 떴습니다! {target_pill} 박스를 다시 그려주세요.")
# print("👉 박스 그리고 's' 누르면 저장됩니다.")

# while True:
#     cv2.imshow(win_name, img)
#     key = cv2.waitKey(10) & 0xFF
    
#     # 윈도우 X버튼 감지
#     if cv2.getWindowProperty(win_name, cv2.WND_PROP_VISIBLE) < 1:
#         break

#     if key == ord('s'): 
#         if not bbox:
#             print("⚠️ 박스를 먼저 그려주세요!")
#             continue
        
#         os.makedirs(os.path.dirname(save_path), exist_ok=True)
#         json_data = {
#             "file_name": os.path.basename(img_path),
#             "bbox": bbox, 
#             "drug_id": target_pill
#         }
        
#         # 무조건 덮어쓰기
#         with open(save_path, 'w', encoding='utf-8') as f:
#             json.dump(json_data, f, indent=4, ensure_ascii=False)
            
#         print(f"💾 {target_pill} 수정 완료! 프로그램이 종료됩니다.")
#         break

#     elif key == ord('r'):
#         img = original_img.copy()
#         bbox = []
        
#     elif key == ord('q'):
#         break
        
# cv2.destroyAllWindows()

=== 🚑 긴급 복구 모드: K-020014 ===
📸 창이 떴습니다! K-020014 박스를 다시 그려주세요.
👉 박스 그리고 's' 누르면 저장됩니다.


최종 검수

In [7]:
import cv2
import json
import os
import numpy as np

# ==========================================
# [설정] 경로가 맞는지 꼭 확인하세요!
# ==========================================
base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data" 

# 작업 대상 리스트
missing_files = [
    { "img_rel": r"train_images\K-003351-013900-021325_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-013900-021325_json\K-003351\K-003351-013900-021325_0_2_0_2_70_000_200.json", "target_pill": "K-003351" },
    { "img_rel": r"train_images\K-003351-013900-036637_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-013900-036637_json\K-003351\K-003351-013900-036637_0_2_0_2_70_000_200.json", "target_pill": "K-003351" },
    { "img_rel": r"train_images\K-003351-020014-022074_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003351-020014-022074_json\K-020014\K-003351-020014-022074_0_2_0_2_90_000_200.json", "target_pill": "K-020014" },
    { "img_rel": r"train_images\K-003351-021325-032310_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003351-021325-032310_json\K-032310\K-003351-021325-032310_0_2_0_2_90_000_200.json", "target_pill": "K-032310" },
    { "img_rel": r"train_images\K-003351-032310-038162_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-032310-038162_json\K-003351\K-003351-032310-038162_0_2_0_2_70_000_200.json", "target_pill": "K-003351" },
    { "img_rel": r"train_images\K-003351-033880-038162_0_2_0_2_75_000_200.png", "json_rel": r"train_annotations\K-003351-033880-038162_json\K-033880\K-003351-033880-038162_0_2_0_2_75_000_200.json", "target_pill": "K-033880" },
    { "img_rel": r"train_images\K-003351-035206-041768_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-035206-041768_json\K-003351\K-003351-035206-041768_0_2_0_2_70_000_200.json", "target_pill": "K-003351" },
    { "img_rel": r"train_images\K-003544-004543-012247-016548_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003544-004543-012247-016548_json\K-004543\K-003544-004543-012247-016548_0_2_0_2_90_000_200.json", "target_pill": "K-004543" }
]

print("=== 🕵️‍♀️ 최종 결과물 검수를 시작합니다 ===")
print("키보드 '스페이스바' 또는 '엔터'를 누르면 다음 장으로 넘어갑니다.")

success_count = 0

for i, item in enumerate(missing_files):
    img_path = os.path.join(base_path, item["img_rel"])
    json_path = os.path.join(base_path, item["json_rel"])
    target = item["target_pill"]
    
    # 1. JSON 파일 확인
    if not os.path.exists(json_path):
        print(f"❌ [{i+1}/8] JSON 파일 없음!! : {target}")
        continue
        
    # 2. JSON 내용 읽기
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        bbox = data.get("bbox", [])
    
    # 3. 박스 유효성 검사 (길이가 0이 아닌지)
    if not bbox or len(bbox) != 4 or bbox[2] < 1 or bbox[3] < 1:
        print(f"⚠️ [{i+1}/8] 박스 데이터가 비정상입니다! (크기 0): {target}")
        continue

    # 4. 이미지 불러오기
    try:
        img_array = np.fromfile(img_path, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    except:
        print(f"❌ 이미지 로드 실패: {img_path}")
        continue

    # 5. 파란색 박스 그리기
    x, y, w, h = bbox
    cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2) # 파란색
    
    # 텍스트 라벨
    text = f"{target} (OK)"
    cv2.putText(img, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    # 6. 보여주기
    win_name = f"Final Check {i+1}/8"
    cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)
    cv2.imshow(win_name, img)
    
    print(f"✅ [{i+1}/8] {target} 확인 중... (bbox: {bbox})")
    cv2.waitKey(0) # 키 입력 대기
    cv2.destroyWindow(win_name)
    
    success_count += 1

cv2.destroyAllWindows()
print("------------------------------------------------")
if success_count == 8:
    print(f"🎉 축하합니다! 8개 파일 모두 정상적으로 생성되었습니다.")
    print("이제 깃허브에 올리시면 됩니다!")
else:
    print(f"😱 총 8개 중 {success_count}개만 확인되었습니다.")
    print("터미널 위쪽 로그를 확인해서 ❌나 ⚠️가 뜬 파일을 다시 작업해주세요.")

=== 🕵️‍♀️ 최종 결과물 검수를 시작합니다 ===
키보드 '스페이스바' 또는 '엔터'를 누르면 다음 장으로 넘어갑니다.
✅ [1/8] K-003351 확인 중... (bbox: [393, 838, 176, 184])
✅ [2/8] K-003351 확인 중... (bbox: [432, 868, 183, 187])
✅ [3/8] K-020014 확인 중... (bbox: [64, 714, 317, 329])
✅ [4/8] K-032310 확인 중... (bbox: [578, 815, 368, 249])
✅ [5/8] K-003351 확인 중... (bbox: [380, 849, 189, 198])
✅ [6/8] K-033880 확인 중... (bbox: [71, 590, 285, 435])
✅ [7/8] K-003351 확인 중... (bbox: [457, 870, 176, 192])
✅ [8/8] K-004543 확인 중... (bbox: [637, 192, 201, 201])
------------------------------------------------
🎉 축하합니다! 8개 파일 모두 정상적으로 생성되었습니다.
이제 깃허브에 올리시면 됩니다!


JSON 형식 변환 코드

In [1]:
import cv2
import json
import os
import glob

# ==========================================
# [설정] 경로가 맞는지 다시 확인해주세요!
# ==========================================
base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data" 

# 변환 대상 파일 목록 (아까 작업한 8개 파일)
target_files = [
    { "img_rel": r"train_images\K-003351-013900-021325_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-013900-021325_json\K-003351\K-003351-013900-021325_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003351-013900-036637_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-013900-036637_json\K-003351\K-003351-013900-036637_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003351-020014-022074_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003351-020014-022074_json\K-020014\K-003351-020014-022074_0_2_0_2_90_000_200.json" },
    { "img_rel": r"train_images\K-003351-021325-032310_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003351-021325-032310_json\K-032310\K-003351-021325-032310_0_2_0_2_90_000_200.json" },
    { "img_rel": r"train_images\K-003351-032310-038162_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-032310-038162_json\K-003351\K-003351-032310-038162_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003351-033880-038162_0_2_0_2_75_000_200.png", "json_rel": r"train_annotations\K-003351-033880-038162_json\K-033880\K-003351-033880-038162_0_2_0_2_75_000_200.json" },
    { "img_rel": r"train_images\K-003351-035206-041768_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-035206-041768_json\K-003351\K-003351-035206-041768_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003544-004543-012247-016548_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003544-004543-012247-016548_json\K-004543\K-003544-004543-012247-016548_0_2_0_2_90_000_200.json" }
]

print("=== JSON 포맷 업그레이드를 시작합니다 ===")

for i, item in enumerate(target_files):
    img_path = os.path.join(base_path, item["img_rel"])
    json_path = os.path.join(base_path, item["json_rel"])
    
    if not os.path.exists(json_path):
        print(f"⚠️ 파일 없음: {item['json_rel']}")
        continue

    # 1. 기존 심플 JSON 읽기
    with open(json_path, 'r', encoding='utf-8') as f:
        simple_data = json.load(f)
    
    # 2. 이미지 정보 읽기 (Width, Height 필수!)
    img = cv2.imread(img_path)
    if img is None:
        print(f"❌ 이미지 로드 실패: {img_path}")
        h, w = 0, 0 # 실패시 기본값
    else:
        h, w, _ = img.shape
    
    drug_id = simple_data.get("drug_id", "Unknown")
    bbox = simple_data.get("bbox", [0,0,0,0])
    file_name = simple_data.get("file_name", os.path.basename(img_path))
    
    # ID 숫자만 추출 (예: K-003351 -> 3351)
    try:
        numeric_id = int(drug_id.split('-')[1])
    except:
        numeric_id = 0 # 예외 처리

    # 3. 복잡한 구조로 변환 (Standard Format)
    complex_data = {
        "images": [
            {
                "file_name": file_name,
                "width": w,
                "height": h,
                "imgfile": file_name,
                "drug_N": drug_id, # ID 입력
                "drug_S": "",      # 상세 정보 없음
                "back_color": "",
                "drug_dir": "",
                "light_color": "",
                "camera_la": 0,
                "camera_lo": 0,
                "size": 0,
                "dl_idx": str(numeric_id),
                "dl_mapping_code": drug_id,
                "dl_name": "",     # 상세 정보 없음
                "dl_name_en": "",
                "img_key": "",
                "dl_material": "",
                "dl_material_en": "",
                "dl_custom_shape": "",
                "dl_company": "",
                "dl_company_en": "",
                "di_company_mf": "",
                "di_company_mf_en": "",
                "item_seq": 0,
                "di_item_permit_date": "",
                "di_class_no": "",
                "di_etc_otc_code": "",
                "di_edi_code": "",
                "chart": "",
                "drug_shape": "",
                "thick": 0,
                "leng_long": 0,
                "leng_short": 0,
                "print_front": "",
                "print_back": "",
                "color_class1": "",
                "color_class2": "",
                "line_front": "",
                "line_back": "",
                "img_regist_ts": "",
                "form_code_name": "",
                "mark_code_front_anal": "",
                "mark_code_back_anal": "",
                "mark_code_front_img": "",
                "mark_code_back_img": "",
                "mark_code_front": "",
                "mark_code_back": "",
                "change_date": "",
                "id": numeric_id # 임시 ID
            }
        ],
        "type": "instances",
        "annotations": [
            {
                "area": bbox[2] * bbox[3], # 가로 * 세로
                "iscrowd": 0,
                "bbox": bbox, # 우리가 찾은 좌표
                "category_id": numeric_id,
                "ignore": 0,
                "segmentation": [],
                "id": numeric_id, # 임시 ID
                "image_id": numeric_id # 임시 ID
            }
        ],
        "categories": [
            {
                "supercategory": "pill",
                "id": numeric_id,
                "name": drug_id # 예: K-003351
            }
        ]
    }
    
    # 4. 덮어쓰기 저장
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(complex_data, f, indent=4, ensure_ascii=False)
        
    print(f"✅ 변환 완료: {file_name}")

print("🎉 모든 파일의 형식을 맞췄습니다!")

=== JSON 포맷 업그레이드를 시작합니다 ===
✅ 변환 완료: K-003351-013900-021325_0_2_0_2_70_000_200.png
✅ 변환 완료: K-003351-013900-036637_0_2_0_2_70_000_200.png
✅ 변환 완료: K-003351-020014-022074_0_2_0_2_90_000_200.png
✅ 변환 완료: K-003351-021325-032310_0_2_0_2_90_000_200.png
✅ 변환 완료: K-003351-032310-038162_0_2_0_2_70_000_200.png
✅ 변환 완료: K-003351-033880-038162_0_2_0_2_75_000_200.png
✅ 변환 완료: K-003351-035206-041768_0_2_0_2_70_000_200.png
✅ 변환 완료: K-003544-004543-012247-016548_0_2_0_2_90_000_200.png
🎉 모든 파일의 형식을 맞췄습니다!


알약 정보 복제 코드

In [2]:
import cv2
import json
import os
import glob

# ==========================================
# [설정] 경로 확인 필수!
# ==========================================
base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data" 

target_files = [
    { "img_rel": r"train_images\K-003351-013900-021325_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-013900-021325_json\K-003351\K-003351-013900-021325_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003351-013900-036637_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-013900-036637_json\K-003351\K-003351-013900-036637_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003351-020014-022074_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003351-020014-022074_json\K-020014\K-003351-020014-022074_0_2_0_2_90_000_200.json" },
    { "img_rel": r"train_images\K-003351-021325-032310_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003351-021325-032310_json\K-032310\K-003351-021325-032310_0_2_0_2_90_000_200.json" },
    { "img_rel": r"train_images\K-003351-032310-038162_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-032310-038162_json\K-003351\K-003351-032310-038162_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003351-033880-038162_0_2_0_2_75_000_200.png", "json_rel": r"train_annotations\K-003351-033880-038162_json\K-033880\K-003351-033880-038162_0_2_0_2_75_000_200.json" },
    { "img_rel": r"train_images\K-003351-035206-041768_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-035206-041768_json\K-003351\K-003351-035206-041768_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003544-004543-012247-016548_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003544-004543-012247-016548_json\K-004543\K-003544-004543-012247-016548_0_2_0_2_90_000_200.json" }
]

# [중요] Images 정보 중 절대 덮어쓰면 안 되는 내 고유 정보
# (이 정보들은 내 파일에 있는 값을 그대로 유지합니다)
KEEP_MY_KEYS_IMG = ["file_name", "width", "height", "imgfile", "id"]

print("=== 🧬 알약 정보 복제 (Annotations 보존 모드) ===")

for i, item in enumerate(target_files):
    my_json_path = os.path.join(base_path, item["json_rel"])
    folder_path = os.path.dirname(my_json_path)
    
    if not os.path.exists(my_json_path):
        print(f"⚠️ 내 파일이 없어요: {item['json_rel']}")
        continue

    # 1. 내 파일 읽기
    with open(my_json_path, 'r', encoding='utf-8') as f:
        my_data = json.load(f)

    # 2. 족보(Donor) 파일 찾기
    donor_data = None
    candidate_files = glob.glob(os.path.join(folder_path, "*.json"))
    
    for candidate in candidate_files:
        if os.path.abspath(candidate) == os.path.abspath(my_json_path):
            continue
        try:
            with open(candidate, 'r', encoding='utf-8') as f:
                temp_data = json.load(f)
                # 족보가 정상적인지 확인 (images와 categories가 꽉 차 있는지)
                if ("images" in temp_data and "dl_company" in temp_data["images"][0] and
                    "categories" in temp_data and len(temp_data["categories"]) > 0):
                    donor_data = temp_data
                    break
        except:
            continue
    
    if donor_data is None:
        print(f"⚠️ [{i+1}] 족보 없음 (패스): {item['target_pill']}")
        continue

    # 3. 데이터 복사 (Categories & Images 만 수행)
    
    # (1) Categories: 100% 복사 (알약 이름, 코드 등)
    #     같은 알약이므로 정보가 완전히 동일해야 함
    my_data["categories"] = donor_data["categories"]

    # (2) Images: 족보 정보를 가져오되, 내 파일 고유 정보는 유지
    #     - 족보(Donor)의 이미지 정보를 복사해 옴 (제조사, 성분 등)
    new_img_info = donor_data["images"][0].copy()
    
    #     - 내 파일(My)에 있던 고유 정보(파일명, 크기, ID)로 다시 덮어씀
    for key in KEEP_MY_KEYS_IMG:
        if key in my_data["images"][0]:
            new_img_info[key] = my_data["images"][0][key]
            
    #     - 합쳐진 정보를 내 파일에 적용
    my_data["images"][0] = new_img_info

    # (3) Annotations: [중요] 건드리지 않음!
    #     - 기존 코드에 있던 annotations 수정 부분은 모두 삭제했습니다.
    #     - 현재 가지고 계신 bbox, area, id가 그대로 보존됩니다.

    # 4. 저장하기
    with open(my_json_path, 'w', encoding='utf-8') as f:
        json.dump(my_data, f, indent=4, ensure_ascii=False)
        
    print(f"✅ [{i+1}] 메타데이터 업데이트 완료 (Annotations 유지됨)")

print("\n🎉 모든 작업이 안전하게 완료되었습니다!")

=== 🧬 알약 정보 복제 (Annotations 보존 모드) ===
✅ [1] 메타데이터 업데이트 완료 (Annotations 유지됨)
✅ [2] 메타데이터 업데이트 완료 (Annotations 유지됨)
✅ [3] 메타데이터 업데이트 완료 (Annotations 유지됨)
✅ [4] 메타데이터 업데이트 완료 (Annotations 유지됨)
✅ [5] 메타데이터 업데이트 완료 (Annotations 유지됨)
✅ [6] 메타데이터 업데이트 완료 (Annotations 유지됨)
✅ [7] 메타데이터 업데이트 완료 (Annotations 유지됨)
✅ [8] 메타데이터 업데이트 완료 (Annotations 유지됨)

🎉 모든 작업이 안전하게 완료되었습니다!


원본 ID 분석기 (판다스 없음 Ver.)

In [4]:
import json
import os
import glob
import csv

# ==========================================
# [설정] 데이터 경로
# ==========================================
base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data" 

# 제외할 파일 명단 (우리가 작업 중인 8개 파일)
excluded_filenames = [
    "K-003351-013900-021325_0_2_0_2_70_000_200.json",
    "K-003351-013900-036637_0_2_0_2_70_000_200.json",
    "K-003351-020014-022074_0_2_0_2_90_000_200.json",
    "K-003351-021325-032310_0_2_0_2_90_000_200.json",
    "K-003351-032310-038162_0_2_0_2_70_000_200.json",
    "K-003351-033880-038162_0_2_0_2_75_000_200.json",
    "K-003351-035206-041768_0_2_0_2_70_000_200.json",
    "K-003544-004543-012247-016548_0_2_0_2_90_000_200.json"
]

def analyze_json_ids_no_pandas():
    print("📂 JSON 파일들을 검색 중입니다 (No Pandas Mode)...")
    search_pattern = os.path.join(base_path, "train_annotations", "**", "*.json")
    json_files = glob.glob(search_pattern, recursive=True)
    
    data_list = []
    skipped_count = 0
    
    print(f"   -> 총 {len(json_files)}개 파일 발견됨.")

    for filepath in json_files:
        file_name = os.path.basename(filepath)
        
        # 제외 명단 확인
        if file_name in excluded_filenames:
            skipped_count += 1
            continue
            
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # (1) Images ID
            if "images" in data and len(data["images"]) > 0:
                img_id = data["images"][0].get("id", -1)
            else:
                img_id = -1
            
            # (2) Annotations ID
            if "annotations" in data and len(data["annotations"]) > 0:
                ann_id = data["annotations"][0].get("id", "None")
                ann_image_id = data["annotations"][0].get("image_id", "None")
            else:
                ann_id = "None"
                ann_image_id = "None"

            data_list.append({
                "File Name": file_name,
                "Images ID": img_id,
                "Annotations ID": ann_id,
                "Annotations Image_ID": ann_image_id
            })
            
        except Exception as e:
            continue

    if not data_list:
        print("❌ 분석할 원본 파일이 없습니다.")
        return

    # 정렬 (Images ID 기준 오름차순) - 파이썬 기본 기능 사용
    # 숫자가 아닌 경우 -1로 취급하여 정렬
    def sort_key(item):
        try:
            return int(item["Images ID"])
        except:
            return -1

    data_list.sort(key=sort_key)

    # 4. 바탕화면에 CSV 저장 (엑셀에서 열림)
    desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
    save_path = os.path.join(desktop_path, "original_id_check.csv")
    
    try:
        with open(save_path, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=["File Name", "Images ID", "Annotations ID", "Annotations Image_ID"])
            writer.writeheader()
            writer.writerows(data_list)
            
        print("\n" + "="*50)
        print(f"✅ [CSV 저장 완료] 엑셀에서 열어보세요!")
        print(f"   - 제외한 파일 수: {skipped_count}개")
        print(f"   - 분석한 파일 수: {len(data_list)}개")
        print(f"📁 파일 위치: {save_path}")
        print("="*50)
    except Exception as e:
        print(f"❌ 저장 실패: {e}")

if __name__ == "__main__":
    analyze_json_ids_no_pandas()

📂 JSON 파일들을 검색 중입니다 (No Pandas Mode)...
   -> 총 771개 파일 발견됨.

✅ [CSV 저장 완료] 엑셀에서 열어보세요!
   - 제외한 파일 수: 25개
   - 분석한 파일 수: 746개
📁 파일 위치: C:\Users\qwer0\Desktop\original_id_check.csv


iscrowd값 조사

In [3]:
import json
import os
import glob

# ==========================================
# [설정] 데이터 경로
# ==========================================
base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data" 

def check_iscrowd_values():
    print("🕵️‍♀️ iscrowd 값을 전수 조사합니다...")
    
    search_pattern = os.path.join(base_path, "train_annotations", "**", "*.json")
    json_files = glob.glob(search_pattern, recursive=True)
    
    count_0 = 0
    count_1 = 0
    files_with_1 = []

    for filepath in json_files:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            if "annotations" in data and len(data["annotations"]) > 0:
                # 첫 번째 어노테이션의 iscrowd 값 확인
                val = data["annotations"][0].get("iscrowd", -1)
                
                if val == 0:
                    count_0 += 1
                elif val == 1:
                    count_1 += 1
                    files_with_1.append(os.path.basename(filepath))
                else:
                    pass # 값이 없거나 이상한 경우
        except:
            continue

    print("\n" + "="*40)
    print(f"📊 분석 결과 (총 {count_0 + count_1}개 파일)")
    print(f"   - iscrowd = 0 (단일 객체): {count_0}개")
    print(f"   - iscrowd = 1 (군중/무시): {count_1}개")
    print("="*40)

    if count_1 > 0:
        print(f"⚠️ 주의! iscrowd가 1인 파일이 {count_1}개 발견되었습니다.")
        print("목록:", files_with_1[:5], "...")
    else:
        print("✅ 결론: 모든 파일이 0입니다. 안심하고 0을 쓰세요!")

if __name__ == "__main__":
    check_iscrowd_values()

🕵️‍♀️ iscrowd 값을 전수 조사합니다...

📊 분석 결과 (총 771개 파일)
   - iscrowd = 0 (단일 객체): 771개
   - iscrowd = 1 (군중/무시): 0개
✅ 결론: 모든 파일이 0입니다. 안심하고 0을 쓰세요!


id 확인

In [5]:
import json
import os
import glob

# ==========================================
# [설정] 경로 확인
# ==========================================
base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data" 

target_files = [
    { "img_rel": r"train_images\K-003351-013900-021325_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-013900-021325_json\K-003351\K-003351-013900-021325_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003351-013900-036637_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-013900-036637_json\K-003351\K-003351-013900-036637_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003351-020014-022074_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003351-020014-022074_json\K-020014\K-003351-020014-022074_0_2_0_2_90_000_200.json" },
    { "img_rel": r"train_images\K-003351-021325-032310_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003351-021325-032310_json\K-032310\K-003351-021325-032310_0_2_0_2_90_000_200.json" },
    { "img_rel": r"train_images\K-003351-032310-038162_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-032310-038162_json\K-003351\K-003351-032310-038162_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003351-033880-038162_0_2_0_2_75_000_200.png", "json_rel": r"train_annotations\K-003351-033880-038162_json\K-033880\K-003351-033880-038162_0_2_0_2_75_000_200.json" },
    { "img_rel": r"train_images\K-003351-035206-041768_0_2_0_2_70_000_200.png", "json_rel": r"train_annotations\K-003351-035206-041768_json\K-003351\K-003351-035206-041768_0_2_0_2_70_000_200.json" },
    { "img_rel": r"train_images\K-003544-004543-012247-016548_0_2_0_2_90_000_200.png", "json_rel": r"train_annotations\K-003544-004543-012247-016548_json\K-004543\K-003544-004543-012247-016548_0_2_0_2_90_000_200.json" }
]

print("=== 🔍 [검증 모드] 파일 변경 없이 ID 확인만 수행합니다 ===")

for i, item in enumerate(target_files):
    json_path = os.path.join(base_path, item["json_rel"])
    folder_path = os.path.dirname(json_path)
    file_name = os.path.basename(item["img_rel"])
    
    # 키 추출 (예: K-003351-013900-021325)
    pill_key = file_name.split('_')[0]
    
    # 형님 찾기
    sibling_id = "없음"
    sibling_name = "없음"
    
    if os.path.exists(folder_path):
        candidate_files = glob.glob(os.path.join(folder_path, "*.json"))
        for candidate in candidate_files:
            # 나 자신은 제외
            if os.path.abspath(candidate) == os.path.abspath(json_path): continue
            
            cand_name = os.path.basename(candidate)
            if cand_name.startswith(pill_key):
                try:
                    with open(candidate, 'r', encoding='utf-8') as f:
                        temp = json.load(f)
                        if "images" in temp and len(temp["images"]) > 0:
                            sibling_id = temp["images"][0]["id"]
                            sibling_name = cand_name
                            break
                except: continue

    print(f"[{i+1}] {pill_key} ...")
    if sibling_id != "없음":
        print(f"   ✅ 매칭 성공! -> 형님 파일: {sibling_name}")
        print(f"   💾 가져올 ID: {sibling_id}")
    else:
        print(f"   ⚠️ 매칭 실패! (새 번호 필요)")
    print("-" * 50)

=== 🔍 [검증 모드] 파일 변경 없이 ID 확인만 수행합니다 ===
[1] K-003351-013900-021325 ...
   ✅ 매칭 성공! -> 형님 파일: K-003351-013900-021325_0_2_0_2_75_000_200.json
   💾 가져올 ID: 194
--------------------------------------------------
[2] K-003351-013900-036637 ...
   ✅ 매칭 성공! -> 형님 파일: K-003351-013900-036637_0_2_0_2_75_000_200.json
   💾 가져올 ID: 1268
--------------------------------------------------
[3] K-003351-020014-022074 ...
   ✅ 매칭 성공! -> 형님 파일: K-003351-020014-022074_0_2_0_2_70_000_200.json
   💾 가져올 ID: 781
--------------------------------------------------
[4] K-003351-021325-032310 ...
   ✅ 매칭 성공! -> 형님 파일: K-003351-021325-032310_0_2_0_2_75_000_200.json
   💾 가져올 ID: 1382
--------------------------------------------------
[5] K-003351-032310-038162 ...
   ✅ 매칭 성공! -> 형님 파일: K-003351-032310-038162_0_2_0_2_90_000_200.json
   💾 가져올 ID: 1434
--------------------------------------------------
[6] K-003351-033880-038162 ...
   ✅ 매칭 성공! -> 형님 파일: K-003351-033880-038162_0_2_0_2_70_000_200.json
   💾 가져올 ID: 1406

어노테이션 id 확인

In [6]:
import json
import os
import glob
import csv

# ==========================================
# [설정] 데이터 경로 (본인 경로에 맞게 수정하세요)
# ==========================================
base_path = r"C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data" 

def create_fresh_csv():
    # 저장할 경로 (바탕화면)
    desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
    output_csv = os.path.join(desktop_path, "current_dataset_status.csv")

    print(f"📂 폴더 스캔 시작: {base_path}")
    print("⏳ 모든 JSON 파일을 읽어들이는 중입니다...")

    # 1. 모든 JSON 파일 찾기
    search_pattern = os.path.join(base_path, "train_annotations", "**", "*.json")
    json_files = glob.glob(search_pattern, recursive=True)

    if not json_files:
        print("❌ JSON 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    # 2. 데이터 추출
    rows = []
    
    for filepath in json_files:
        filename = os.path.basename(filepath)
        
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # (1) 이미지 정보 가져오기
            if "images" in data and len(data["images"]) > 0:
                img_id = data["images"][0].get("id", "None")
                file_name_in_json = data["images"][0].get("file_name", "None")
            else:
                img_id = "없음"
                file_name_in_json = "없음"

            # (2) 어노테이션 정보 가져오기 (박스가 여러 개일 수 있음)
            if "annotations" in data and len(data["annotations"]) > 0:
                for ann in data["annotations"]:
                    ann_id = ann.get("id", "None")
                    cat_id = ann.get("category_id", "None")
                    bbox = ann.get("bbox", [])
                    
                    # bbox가 리스트인지 확인하고 문자열로 변환 (CSV 저장을 위해)
                    bbox_str = str(bbox) if isinstance(bbox, list) else "Invalid"

                    rows.append({
                        "JSON File": filename,
                        "Image File": file_name_in_json,
                        "Image ID": img_id,
                        "Category ID": cat_id,
                        "Annotation ID": ann_id,
                        "BBox": bbox_str
                    })
            else:
                # 어노테이션이 없는 경우 (빈 껍데기 파일)
                rows.append({
                    "JSON File": filename,
                    "Image File": file_name_in_json,
                    "Image ID": img_id,
                    "Category ID": "없음",
                    "Annotation ID": "없음",
                    "BBox": "없음"
                })

        except Exception as e:
            print(f"⚠️ 읽기 실패 ({filename}): {e}")
            continue

    # 3. CSV로 저장 (ID 순서대로 정렬해서)
    # 정렬 기준: Image ID가 숫자면 숫자순, 아니면 문자순
    def sort_key(row):
        try:
            return int(row["Annotation ID"])
        except:
            return 999999999 # 숫자가 아니면 맨 뒤로

    rows.sort(key=sort_key)

    try:
        with open(output_csv, 'w', newline='', encoding='utf-8-sig') as f:
            # 헤더 정의
            fieldnames = ["JSON File", "Image File", "Image ID", "Category ID", "Annotation ID", "BBox"]
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            
            writer.writeheader()
            writer.writerows(rows)

        print("\n" + "="*50)
        print(f"✅ CSV 생성 완료!")
        print(f"📄 파일명: current_dataset_status.csv")
        print(f"📁 저장 위치: {desktop_path}")
        print(f"📊 총 박스(행) 개수: {len(rows)}개")
        print("="*50)

    except Exception as e:
        print(f"❌ 저장 실패: {e}")

if __name__ == "__main__":
    create_fresh_csv()

📂 폴더 스캔 시작: C:\Users\qwer0\Documents\GitHub\sprintAI07_git\AI_07_basic\data
⏳ 모든 JSON 파일을 읽어들이는 중입니다...

✅ CSV 생성 완료!
📄 파일명: current_dataset_status.csv
📁 저장 위치: C:\Users\qwer0\Desktop
📊 총 박스(행) 개수: 771개
